# Principal Component Analysis

Implement PCA from scratch (covariance eigendecomposition and SVD), validate against scikit-learn, and visualise a 2-D projection and the cumulative explained-variance curve.

## Configuration

Device, seed, and dtype come from `config.toml` via `shared.config.configure()`.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe under nbconvert
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)


running on: mps


## Dataset

We create a 2-D dataset with strong correlation along one direction so PCA captures most variance in the first component.  We also add a 3rd uncorrelated dimension, giving us a dataset in R^3 that we will project to R^2.

In [2]:
import numpy as np

rng = np.random.default_rng(42)

# 200 samples, 3 features; first two features are strongly correlated
N, D = 200, 3
t = rng.standard_normal(N)
X_np = np.column_stack([
    2.0 * t + 0.3 * rng.standard_normal(N),
    1.8 * t + 0.3 * rng.standard_normal(N),
    rng.standard_normal(N),
])
print("X shape:", X_np.shape, "  mean:", X_np.mean(axis=0).round(3))


X shape: (200, 3)   mean: [-0.055 -0.075 -0.037]


## PCA from Scratch — Covariance Eigendecomposition

Steps:
1. Center the data by subtracting the column mean.
2. Compute the empirical covariance matrix Σ = (1/n) Xᶜᵀ Xᶜ.
3. Eigendecompose Σ; sort eigenvalues descending — the eigenvectors are the principal directions.
4. Project: Z = Xᶜ · V_k.
5. Explained-variance ratio: λ_j / Σ_k λ_k.

> **Note:** `torch.linalg.eigh` requires `.cpu()` on MPS backends (Apple Silicon) because MPS does not yet support all linalg ops.  We always call it on a CPU copy.

In [3]:
def pca_eig(X: np.ndarray, k: int):
    """PCA via covariance eigendecomposition.

    Args:
        X: (n, d) data matrix.
        k: Number of principal components to keep.

    Returns:
        components: (k, d) principal directions (rows), sorted by descending variance.
        explained_variance_ratio: (k,) fraction of total variance per component.
        Z: (n, k) projected data.
        X_mean: (d,) column means used for centering.
    """
    # Work in float64 for numerical stability
    X_t = torch.from_numpy(X).double()  # keep on CPU — eigh not supported on MPS
    X_mean = X_t.mean(dim=0)
    Xc = X_t - X_mean  # centered: (n, d)

    # Covariance matrix (use 1/n convention to match sklearn when bias=True)
    n = Xc.shape[0]
    cov = (Xc.T @ Xc) / n  # (d, d)

    # Eigendecomposition — always on CPU copy (MPS rejects linalg.eigh)
    cov_cpu = cov.cpu()
    eigenvalues, eigenvectors = torch.linalg.eigh(cov_cpu)  # eigenvalues ascending

    # Sort descending
    idx = torch.argsort(eigenvalues, descending=True)
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]  # (d, d); columns are eigenvectors

    # Explained variance ratio
    evr = eigenvalues / eigenvalues.sum()  # (d,)

    # Take top-k components; shape (k, d) — rows are principal directions
    components = eigenvectors[:, :k].T  # (k, d)

    # Project centered data
    Z = Xc @ eigenvectors[:, :k]  # (n, k)

    return (
        components.numpy(),
        evr[:k].numpy(),
        Z.numpy(),
        X_mean.numpy(),
        eigenvalues.numpy(),  # all eigenvalues for full EVR curve
        evr.numpy(),          # full EVR
    )


k = 2
components_eig, evr_eig, Z_eig, X_mean_eig, all_eigenvalues, all_evr = pca_eig(X_np, k)

print("Components (rows are principal directions):")
print(components_eig.round(4))
print("Explained variance ratio (top k):", evr_eig.round(4))
print("Cumulative EVR:", np.cumsum(all_evr).round(4))


Components (rows are principal directions):
[[-0.7422 -0.6701  0.0105]
 [-0.0128  0.0298  0.9995]]
Explained variance ratio (top k): [0.8329 0.1539]
Cumulative EVR: [0.8329 0.9868 1.    ]


## PCA from Scratch — SVD Connection

For the centered matrix Xᶜ = U S Vᵀ, the right singular vectors (rows of Vᵀ) are the principal directions, and the singular values relate to explained variance by

```
explained_variance_j = s_j² / n
explained_variance_ratio_j = s_j² / Σ_k s_k²
```

This is numerically more stable than forming Σ explicitly and is exactly what `sklearn.decomposition.PCA` does.

In [4]:
def pca_svd(X: np.ndarray, k: int):
    """PCA via truncated SVD of the centered data matrix.

    Args:
        X: (n, d) data matrix.
        k: Number of principal components.

    Returns:
        components: (k, d) principal directions (rows).
        explained_variance_ratio: (k,) fraction of variance per component.
        Z: (n, k) projected data.
        X_mean: (d,) column means.
    """
    X_t = torch.from_numpy(X).double()  # CPU — SVD on MPS may not be available
    X_mean = X_t.mean(dim=0)
    Xc = X_t - X_mean

    # Full SVD on CPU copy
    U, S, Vh = torch.linalg.svd(Xc.cpu(), full_matrices=False)  # Vh: (d, d)

    n = Xc.shape[0]
    explained_variance = (S ** 2) / n  # (d,)
    evr = explained_variance / explained_variance.sum()  # (d,)

    # Top-k components: rows of Vh
    components = Vh[:k]  # (k, d)
    Z = Xc.cpu() @ Vh[:k].T  # (n, k)

    return (
        components.numpy(),
        evr[:k].numpy(),
        Z.numpy(),
        X_mean.numpy(),
        evr.numpy(),  # full EVR
    )


components_svd, evr_svd, Z_svd, X_mean_svd, all_evr_svd = pca_svd(X_np, k)

print("SVD components:")
print(components_svd.round(4))
print("SVD explained variance ratio:", evr_svd.round(4))


SVD components:
[[-0.7422 -0.6701  0.0105]
 [ 0.0128 -0.0298 -0.9995]]
SVD explained variance ratio: [0.8329 0.1539]


## Validation Against scikit-learn

Components are unique only up to sign (flipping a direction gives another valid basis vector). We therefore check `|scratch_component · sklearn_component| ≈ 1` for each component and that `explained_variance_ratio_` matches to within `atol=1e-5`.

In [5]:
from sklearn.decomposition import PCA

pca_sk = PCA(n_components=k)
pca_sk.fit(X_np)

sk_components = pca_sk.components_  # (k, d)
sk_evr = pca_sk.explained_variance_ratio_  # (k,)

print("sklearn components:")
print(sk_components.round(4))
print("sklearn EVR:", sk_evr.round(4))

# --- Assert: components match up to sign ---
for i in range(k):
    dot_eig = abs(float(np.dot(components_eig[i], sk_components[i])))
    dot_svd = abs(float(np.dot(components_svd[i], sk_components[i])))
    assert dot_eig > 0.9999, f"Eig component {i} does not align with sklearn: dot={dot_eig:.6f}"
    assert dot_svd > 0.9999, f"SVD component {i} does not align with sklearn: dot={dot_svd:.6f}"

# --- Assert: explained_variance_ratio matches ---
# sklearn uses 1/(n-1) convention; our eig used 1/n; use SVD result which matches sklearn
# sklearn.PCA uses full SVD on centered data (1/(n-1) for explained_variance, but the ratio
# cancels the denominator, so it equals S_j^2 / sum S^2 — same as our SVD EVR)
np.testing.assert_allclose(
    evr_svd, sk_evr, atol=1e-5,
    err_msg="SVD EVR does not match sklearn EVR"
)
print("All assertions passed: components align and EVR matches sklearn.")


sklearn components:
[[ 0.7422  0.6701 -0.0105]
 [-0.0128  0.0298  0.9995]]
sklearn EVR: [0.8329 0.1539]
All assertions passed: components align and EVR matches sklearn.


## Plots

Left: 2-D projection of the data onto the first two principal components.  Right: cumulative explained-variance curve — a useful heuristic for choosing `k`.

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- 2-D projection scatter ---
axes[0].scatter(Z_svd[:, 0], Z_svd[:, 1], s=18, alpha=0.6, edgecolors="none", color="steelblue")
axes[0].set_xlabel(f"PC 1  ({evr_svd[0]*100:.1f}% var)")
axes[0].set_ylabel(f"PC 2  ({evr_svd[1]*100:.1f}% var)")
axes[0].set_title("2-D PCA Projection")
axes[0].axhline(0, color="grey", lw=0.5)
axes[0].axvline(0, color="grey", lw=0.5)

# --- Cumulative EVR curve ---
cum_evr = np.cumsum(all_evr_svd)
axes[1].bar(range(1, len(all_evr_svd) + 1), all_evr_svd, alpha=0.6, label="Individual")
axes[1].plot(range(1, len(cum_evr) + 1), cum_evr, "o-", color="tomato", label="Cumulative")
axes[1].set_xlabel("Principal Component")
axes[1].set_ylabel("Explained Variance Ratio")
axes[1].set_title("Explained Variance per Component")
axes[1].set_xticks(range(1, D + 1))
axes[1].legend()
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("pca_plots.png", dpi=120)
plt.show()
print("Saved pca_plots.png")


Saved pca_plots.png


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_70939/2357711097.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Idiomatic scikit-learn Usage

In production, use `sklearn.decomposition.PCA`.  Below we demonstrate the correct train-only fitting workflow to avoid data leakage: fit PCA **only** on the training split, then `transform` validation data with the learned mean and components.

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Use Iris (4 features → 2 PCs)
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

X_tr, X_val, y_tr, y_val = train_test_split(X_iris, y_iris, test_size=0.25, random_state=0, stratify=y_iris)

# Correct: fit PCA only on training data
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2)),
    ("clf", LogisticRegression(max_iter=1000)),
])
pipe.fit(X_tr, y_tr)

val_acc = pipe.score(X_val, y_val)
print(f"Validation accuracy (Iris, 4→2 PCA + LogReg): {val_acc:.3f}")

pca_step = pipe.named_steps["pca"]
print(f"PCA explained variance ratio: {pca_step.explained_variance_ratio_.round(3)}")
print(f"Cumulative EVR (2 components): {pca_step.explained_variance_ratio_.sum():.3f}")


Validation accuracy (Iris, 4→2 PCA + LogReg): 0.895
PCA explained variance ratio: [0.73  0.232]
Cumulative EVR (2 components): 0.962


## Takeaways

- **PCA = SVD**: Running SVD on the centered data matrix is numerically superior to explicitly forming the covariance matrix.  Both give the same principal directions.
- **Sign ambiguity**: Principal components are unique only up to sign; always compare via absolute dot product.
- **Explained variance ratio** cancels the 1/n vs 1/(n−1) denominator choice, so our SVD-based EVR matches sklearn's exactly.
- **No leakage**: Fit `PCA` (and any scaler) on training data only; transform validation/test data with the learned statistics.
- **When not to use PCA**: When low-variance features are predictive, when interpretability of original features is required, or when the structure is nonlinear (use UMAP/t-SNE for visualization instead).
- **MPS note**: `torch.linalg.eigh` and `torch.linalg.svd` require `.cpu()` tensors on Apple Silicon MPS backends; the overhead is negligible for small matrices.